# Categorical Boosting (CatBoost)

**Notebook 3 of 4.** A tree asks `feature <= threshold`, and `occupation` is not a number,
so notebook 01 one-hot encoded the census text columns before fitting anything. CatBoost is
named for what it does with those columns instead, and that is the subject of this notebook.

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, root_mean_squared_error
from catboost import CatBoostClassifier, CatBoostRegressor
from xgboost import XGBClassifier, XGBRegressor

## 1. The data

`data/census.csv`: 45,222 records from the 1994 US census, with `income` saying whether the
person earned more than $50,000. Thirteen columns describe them and **eight are text**.

How many distinct values a text column takes is its **cardinality**, and that is the number
deciding which encoding is affordable.

In [ ]:
census = pd.read_csv("../data/census.csv")

categorical = [
    "workclass", "education_level", "marital-status", "occupation",
    "relationship", "race", "sex", "native-country",
]

y = (census["income"] == ">50K").astype(int)
X = census.drop(columns=["income"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X[categorical].nunique().sort_values(ascending=False).to_string())

## 2. One-hot, and what it costs

One-hot encoding gives every category a column of its own, holding 1 where the row has that
value and 0 everywhere else.

In [ ]:
# one column per category, so the width grows with the cardinalities above
one_hot = pd.get_dummies(X, columns=categorical).astype(float)

O_train, O_test, _, _ = train_test_split(
    one_hot, y, test_size=0.2, random_state=42, stratify=y
)
print(f"{X.shape[1]} columns became {one_hot.shape[1]}")

xgb = XGBClassifier(n_estimators=300, random_state=42, n_jobs=-1)
xgb.fit(O_train, y_train)
print(f"xgboost: {accuracy_score(y_test, xgb.predict(O_test)):.4f} accuracy, "
      f"{roc_auc_score(y_test, xgb.predict_proba(O_test)[:, 1]):.4f} AUC")

**0.8679**, from 103 columns. One-hot is fine on this table, and its two costs are still
worth naming, because both grow with cardinality:

- **the table gets wider** — `native-country` alone adds 41 columns, nearly all of them
  nearly empty;
- **a tree can ask about one category at a time.** Separating six occupations from the other
  eight takes six splits of a one-hot table and one split of a column holding the occupation
  itself.

Put 50,000 customer ids through `get_dummies` and both costs turn fatal.

## 3. The alternative: a target statistic

Rather than splitting a column into many, replace each category with **one number computed
from the target**: the share of its rows that earned more than $50,000.

In [ ]:
# the share of each occupation earning >50K, from the training rows only
occupation_stat = y_train.groupby(X_train["occupation"]).mean().sort_values(ascending=False)
print(occupation_stat.round(3).to_string())

Fourteen columns collapse into one, ordered in a way a tree can cut: `occupation_stat <=
0.25` separates the lower-paid half in one split. This is **target encoding**, and in this
form it is a trap.

## 4. Why the obvious version is a trap

The number standing in for a row's category was computed **using that row's own target**.
With a hundred rows per category that barely matters; with six, the statistic is mostly the
row itself, and the column quietly carries the answer.

To see how badly that goes, here is a column that means nothing at all: a payroll id drawn
at random, about 6,000 of them.

In [ ]:
rng = np.random.default_rng(42)

# X_train and X_test are left alone: the id lives in separate frames, so section 6
# can still compare the real thirteen columns against their one-hot expansion
train_id = X_train.assign(payroll_id=[f"P{i:05d}" for i in rng.integers(0, 6000, len(X_train))])
test_id = X_test.assign(payroll_id=[f"P{i:05d}" for i in rng.integers(0, 6000, len(X_test))])

print(f"{train_id['payroll_id'].nunique()} ids, "
      f"{train_id['payroll_id'].value_counts().median():.0f} rows each (median)")

# three real columns, to be judged with and without the meaningless fourth
plain = ["age", "education-num", "hours-per-week"]

> **Exercise 1.** Target-encode `payroll_id` the obvious way — map each id to the mean of
> `y_train` for that id — add it to `plain` as a fourth feature and fit an
> `XGBClassifier(n_estimators=300)`. Print the **training** and **test** ROC AUC, then do
> the same with `plain` alone. Test ids unseen in training have no mean; give those the
> overall training rate.

<details><summary>
Click here for a hint...
</summary>

`y_train.groupby(train_id["payroll_id"]).mean()` gives the mapping, `.map(...)` applies it,
and `.fillna(y_train.mean())` covers the unseen ids.

</details>

In [ ]:
# the mapping is built from the training rows, as any encoding must be
id_means = y_train.groupby(train_id["payroll_id"]).mean()
prior = y_train.mean()

encoded_train = train_id[plain].assign(
    payroll=train_id["payroll_id"].map(id_means).fillna(prior)
)
encoded_test = test_id[plain].assign(
    payroll=test_id["payroll_id"].map(id_means).fillna(prior)   # unseen ids get the overall rate
)


def auc_pair(columns_train, columns_test, label):
    """Fit XGBoost on the given columns and print its AUC on both splits.

    Args:
        columns_train: the training features.
        columns_test: the test features, same columns in the same order.
        label: what to call this feature set in the printout.
    """
    model = XGBClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    model.fit(columns_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(columns_train)[:, 1])
    test_auc = roc_auc_score(y_test, model.predict_proba(columns_test)[:, 1])
    print(f"{label:<28} train {train_auc:.4f}   test {test_auc:.4f}")


auc_pair(encoded_train, encoded_test, "with the encoded id")
auc_pair(X_train[plain], X_test[plain], "without it")

<details><summary>
Click here to compare your answer...
</summary>

| features | train AUC | test AUC |
| :-- | --: | --: |
| three real columns + the encoded id | 0.9284 | **0.6578** |
| three real columns | 0.8526 | **0.8089** |

A column of **random noise** raised training AUC by 0.076 and took **0.151 off the test
AUC**. The model was not wasting the column, it was misled by it: within the training set
the encoded id predicts the target almost perfectly, because it was built from it, so the
trees lean on it and neglect what generalizes.

None of this is specific to boosting. Target encoding leaks this way under any model, and
the usual workarounds — smoothing toward the prior, computing it out of fold — patch the
same wound.

</details>

## 5. CatBoost's answer: ordered target statistics

CatBoost's fix, from Prokhorenkova et al. (2017), forbids a row from seeing its own target.
It draws a **random permutation** of the training rows and, for row $i$, computes the
statistic from the rows coming **before** $i$ in that order and no others — an artificial
history, as though the data had arrived one row at a time.

Writing $n_i$ for how many earlier rows share row $i$'s category and $s_i$ for how many of
those had the target 1, the value used for row $i$ is

$$\frac{s_i + a\,p}{n_i + a}$$

with $p$ a prior, the overall rate, and $a$ a weight deciding how fast the statistic trusts
the data over that prior. Early rows have almost no history, so they get something close to
$p$; later rows get something close to the true mean.

The same idea is applied a second time to the gradients, as **ordered boosting**, because
ordinary gradient boosting has a milder form of the same problem.

In [ ]:
# the same meaningless column, this time handed to CatBoost as a category
with_id = plain + ["payroll_id"]

start = time.perf_counter()
catboost = CatBoostClassifier(
    iterations=300,
    random_seed=42,
    verbose=0,                   # otherwise it prints a line per iteration
    allow_writing_files=False,   # otherwise it leaves a catboost_info/ folder behind
)
catboost.fit(train_id[with_id], y_train, cat_features=["payroll_id"])

print(f"train AUC {roc_auc_score(y_train, catboost.predict_proba(train_id[with_id])[:, 1]):.4f}   "
      f"test AUC {roc_auc_score(y_test, catboost.predict_proba(test_id[with_id])[:, 1]):.4f}   "
      f"({time.perf_counter() - start:.0f}s)")

importances = pd.Series(catboost.feature_importances_, index=with_id)
print(importances.round(1).to_string())

# the same model without the nuisance column, so the comparison is like for like
without = CatBoostClassifier(
    iterations=300, random_seed=42, verbose=0, allow_writing_files=False
)
without.fit(X_train[plain], y_train)
print(f"\nsame model, no payroll_id: "
      f"test AUC {roc_auc_score(y_test, without.predict_proba(X_test[plain])[:, 1]):.4f}")

**Test AUC 0.8102**, against **0.8150** for the same CatBoost without the column at all. So
the noise cost it **0.005** — not nothing, but next to the naive encoding's **0.151** it is
a rounding error.

Its training AUC of 0.8356 barely exceeds its test AUC, too: the column never got to inflate
anything. The importances agree, with `payroll_id` last at **15.0** behind `age` at 40.9.

## 6. In practice

With that machinery in place, using CatBoost means naming the columns. No encoding step, and
no `get_dummies` call to keep consistent between training and serving.

One more design choice is worth knowing: CatBoost grows **symmetric**, or oblivious, trees,
where every node at the same depth splits on the same feature and threshold. It is a severe
restriction on each tree, and it buys regularization and very fast prediction — a tree of
depth 6 becomes an index into a 64-entry table.

> **Exercise 2.** Fit a `CatBoostClassifier(iterations=300)` on `X_train` — the thirteen
> original columns, no `payroll_id` — passing `cat_features=categorical`, and compare its
> accuracy and AUC with section 2's one-hot XGBoost on the same thirteen columns expanded to
> 103. Time both fits. Better, faster, or neither?

<details><summary>
Click here for a hint...
</summary>

`cat_features` takes the column names, and the rest of `X_train` is numeric already. Keep
`verbose=0` and `allow_writing_files=False`.

</details>

In [ ]:
start = time.perf_counter()
native = CatBoostClassifier(
    iterations=300, random_seed=42, verbose=0, allow_writing_files=False
)
# the eight text columns go in as text; the five numeric ones are left alone
native.fit(X_train, y_train, cat_features=categorical)
native_time = time.perf_counter() - start

start = time.perf_counter()
# refitting section 2's model only to time it on the same machine
onehot_model = XGBClassifier(n_estimators=300, random_state=42, n_jobs=-1)
onehot_model.fit(O_train, y_train)
onehot_time = time.perf_counter() - start

print(f"catboost, {X_train.shape[1]} raw columns:   "
      f"{accuracy_score(y_test, native.predict(X_test)):.4f} accuracy, "
      f"{roc_auc_score(y_test, native.predict_proba(X_test)[:, 1]):.4f} AUC, "
      f"{native_time:.0f}s")
print(f"xgboost, {O_train.shape[1]} one-hot columns: "
      f"{accuracy_score(y_test, onehot_model.predict(O_test)):.4f} accuracy, "
      f"{roc_auc_score(y_test, onehot_model.predict_proba(O_test)[:, 1]):.4f} AUC, "
      f"{onehot_time:.0f}s")

<details><summary>
Click here to compare your answer...
</summary>

| | accuracy | AUC | fit |
| :-- | --: | --: | --: |
| CatBoost, 13 raw columns | 0.8676 | 0.9255 | about 11s |
| XGBoost, 103 one-hot columns | 0.8679 | 0.9240 | about 2s |

**Neither.** They sit within 0.0003 accuracy of each other, and CatBoost took five times as
long. That is the result to expect, and it is why the mechanism came first: the census
columns top out at 41 categories, which one-hot handles comfortably, so there is nothing
here for ordered target statistics to rescue. What CatBoost removed is the encoding step.

Section 4 is where the difference lives. Raise the cardinality until categories hold a
handful of rows each — customer ids, postcodes, product skus — and one-hot becomes unusable
while naive target encoding turns harmful.

</details>

## 7. Every ensemble so far, on one problem

Each notebook here teaches one library on one kind of problem. This section puts them side
by side on a single regression table — the California districts `02_xgboost.ipynb` uses — so
that what is being compared is the algorithms and not the data underneath them.

Every one of them has a regressor as well as a classifier. The split between the two jobs
across these notebooks is a teaching choice, not a property of the libraries.

In [ ]:
# the regression table from notebook 02, so the comparison happens on one problem
districts = pd.read_csv("../data/california_housing.csv")

house_value = districts["MedHouseVal"]
features = pd.get_dummies(
    districts.drop(columns="MedHouseVal"), columns=["location"]
).astype(float)

# F and v rather than X and y: the census split above is still in scope
F_train, F_test, v_train, v_test = train_test_split(
    features, house_value, test_size=0.2, random_state=42
)


def score_regressor(model, label):
    """Fit a regressor on the training districts, then print test RMSE and fitting time.

    Args:
        model: any unfitted regressor with fit and predict.
        label: what to call it in the printed line.

    """
    start = time.perf_counter()
    model.fit(F_train, v_train)
    elapsed = time.perf_counter() - start

    rmse = root_mean_squared_error(v_test, model.predict(F_test))
    print(f"{label:<18} test RMSE {rmse:.3f}   {elapsed:5.1f}s")


print(f"{len(F_train)} training districts, {features.shape[1]} columns")

> **Exercise 3.** Score these four on the California split with `score_regressor`:
>
> - `RandomForestRegressor(n_estimators=200, max_features=0.3)`
> - `AdaBoostRegressor` on a `DecisionTreeRegressor(max_depth=6)`, 100 estimators, rate 0.1
> - `XGBRegressor(n_estimators=992, learning_rate=0.05)`, the settings early stopping
>   reached in `02_xgboost.ipynb`
> - `CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6)`
>
> Which one comes last, and is that where you expected it? About twenty seconds in all.

<details><summary>
Click here for a hint...
</summary>

`CatBoostRegressor` needs `verbose=0` and `allow_writing_files=False`; `AdaBoostRegressor`
takes its weak learner as `estimator=`.

</details>

In [ ]:
score_regressor(
    RandomForestRegressor(n_estimators=200, max_features=0.3, random_state=42, n_jobs=-1),
    "random forest")

# depth 6 rather than a stump: a stump regressor predicts two values and cannot do better
score_regressor(
    AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=6, random_state=42),
                      n_estimators=100, learning_rate=0.1, random_state=42),
    "adaboost")

# the tree count early stopping arrived at in notebook 02
score_regressor(
    XGBRegressor(n_estimators=992, learning_rate=0.05, random_state=42, n_jobs=-1),
    "xgboost")

score_regressor(
    CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=42,
                      verbose=0, allow_writing_files=False),
    "catboost")

<details><summary>
Click here to compare your answer...
</summary>

| model | test RMSE |
| :-- | --: |
| AdaBoost, depth 6 | 0.603 |
| random forest | 0.472 |
| CatBoost | 0.466 |
| XGBoost | **0.437** |

**AdaBoost comes last, and not narrowly**: 0.603 against the random forest's 0.472. It is
the one boosting model here that does not fit the residual — it reweights rows and combines
them by vote, which is a blunter instrument on a regression target than fitting the error
directly.

**XGBoost and CatBoost finish 0.029 apart**, and `02_xgboost.ipynb` measured the
fold-to-fold spread on this data at about 0.028. A gap the size of the noise is not a
ranking. They are the same algorithm with different engineering, and choosing between them
rests on what these notebooks showed: how the categorical columns look, how large the table
is, and how long a fit takes.

`04_lightgbm.ipynb` is optional, and adds a fifth row to this table.

</details>

## Summary

- **Cardinality decides the encoding.** One-hot turned 13 columns into 103, and would turn
  a column of ids into tens of thousands.
- A **target statistic** replaces a category with the mean target in it: one ordered column
  rather than many binary ones.
- **Computed the obvious way it leaks.** A random id, target-encoded, pushed training AUC to
  0.9284 and test AUC down to 0.6578.
- CatBoost computes it over a **random permutation**, from each row's predecessors only, and
  applies the same idea to the gradients as **ordered boosting**. The same noise column then
  cost it **0.005** of test AUC rather than 0.151.
- On **low-cardinality** data it ties with one-hot XGBoost and fits more slowly. What it
  saved in that comparison is the encoding step; section 4 is where the mechanism pays.

Next, in `04_lightgbm.ipynb`: the same family again, built for speed.
- On **one regression problem** all of these libraries take, AdaBoost finished far behind a
  random forest, while XGBoost and CatBoost finished within the fold-to-fold spread of each
  other.